<a href="https://colab.research.google.com/github/rafaellopesdesa/nsbi-lhc-toolkit/blob/ml4hep_school_tutorial/workshops/ml4hep_tifr_colab/paper_summary/02_SLCP_JANA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 02 — JANA-paper and separate-flow baselines

This notebook produces two deliberately distinct baselines.  **JANA-paper**
runs the pinned algorithm in its isolated legacy TensorFlow/BayesFlow
environment, with simulator input adapted to the fixed nested banks. At 1M,
the minibatch size is increased from 32 to 1024 for GPU efficiency; other
model and training settings are retained. In keeping with upstream, this row uses N training pairs,
the two-row shape bank, the fixed two-row Trainer pilot, and the fixed 300-pair
validation bank, and is labelled N+304 in resource tables.  **Separate flows** loads the nominal
posterior and likelihood ensembles selected in notebook 01.

Both baselines receive the full paired diagnostic suite: posterior-route and
likelihood-route C2ST/MMD, route agreement, predictive closure, importance
efficiency, Bayes-cycle and conditional-normalization checks, and comparison
with the analytic SLCP likelihood.  Each is the direct control for corrections
trained over that same flow base in notebook 03.

Several independent Colab runtimes may run this notebook against the same
Drive artifact root.  Each runtime claims one pending `(budget, ML seed)` shard
at a time and skips shards already being trained by another live runtime.


## Repairing saved-model evaluation (02 and 03)

The pinned BayesFlow version stores its orthogonal rotation matrices as ordinary
Tensors, so they are absent from TensorFlow checkpoints. The inference loader
now reconstructs these matrices with the saved **training seed** before restoring
the trained weights. Previously, fresh random rotations could make all posterior
proposals fall outside the prior and raise `All likelihood-route importance
weights are zero`, even though training had completed successfully.

Rerun this notebook from the setup cell with `LOAD_IF_AVAILABLE=True` and the
same artifact root. No completed flow needs retraining. Old JANA diagnostics
and ratio banks are preserved under recovery names and regenerated once.
JANA correction classifiers trained on the old banks must also be fitted again;
they are preserved separately from the corrected classifiers. Separate-flow
models and their corrections are unaffected. Subsequent runs reuse the repaired
outputs normally. The prior, proposals and importance-weight formula are unchanged.


In [ ]:
# Google Colab setup -- safe to rerun and a no-op outside Colab.
import importlib.util
import json
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as package_version
from pathlib import Path

# Must be set before the first CUDA/PyTorch initialization in this process.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"
BRANCH = "ml4hep_school_tutorial"
USE_DRIVE = os.environ.get("PAPER_SUMMARY_USE_DRIVE", "1") != "0"

def run(*args, env=None):
    subprocess.run([str(arg) for arg in args], check=True, env=env)

def installed_version(distribution):
    try:
        return package_version(distribution)
    except PackageNotFoundError:
        return None

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        default_artifact_root = Path(
            "/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP"
        )
    else:
        default_artifact_root = Path("/content/paper_summary_SLCP_artifacts")

    repository = Path("/content/nsbi-lhc-toolkit")
    if not (repository / ".git").is_dir():
        clone_env = os.environ.copy()
        clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
        run(
            "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
            "--branch", BRANCH, REPO_URL, repository, env=clone_env,
        )
    else:
        run("git", "-C", repository, "remote", "set-url", "origin", REPO_URL)
        run("git", "-C", repository, "fetch", "origin", BRANCH)
        run("git", "-C", repository, "checkout", BRANCH)
        run("git", "-C", repository, "pull", "--ff-only", "origin", BRANCH)
    run(
        "git", "-C", repository, "sparse-checkout", "set", "src",
        "workshops/ml4hep_tifr_colab/paper_summary",
    )
    SOURCE_DIR = repository / "workshops" / "ml4hep_tifr_colab" / "paper_summary"

    # Colab already provides the numerical/ML stack used by these notebooks.
    # Install only the two missing modern-runtime packages normally.  In
    # particular, do not let sbibm pull its historical algorithm dependency
    # tree into the current Colab Python environment (currently Python 3.13).
    modern_requirements = []
    if installed_version("nflows") != "0.14":
        modern_requirements.append("nflows==0.14")
    if importlib.util.find_spec("pyro") is None:
        modern_requirements.append("pyro-ppl")
    if modern_requirements:
        run(sys.executable, "-m", "pip", "install", "-q", *modern_requirements)
    if installed_version("sbibm") != "1.1.0":
        # This is the same Python-3.13-safe installation used by Exercises 9
        # and 10: the SLCP task/metrics need sbibm itself, nflows, and Pyro,
        # but not sbibm's old pinned SBI/algorithm environment.
        run(
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            "sbibm==1.1.0",
        )
else:
    candidates = (
        Path.cwd(),
        Path.cwd() / "paper_summary",
        Path.cwd() / "workshops" / "ml4hep_tifr_colab" / "paper_summary",
    )
    SOURCE_DIR = next(
        (candidate.resolve() for candidate in candidates if (candidate / "config.py").is_file()),
        None,
    )
    if SOURCE_DIR is None:
        raise FileNotFoundError("Cannot locate the paper_summary source directory")
    default_artifact_root = SOURCE_DIR / "artifacts"

source_path = str(SOURCE_DIR)
if source_path not in sys.path:
    sys.path.insert(0, source_path)
os.chdir(SOURCE_DIR)

ARTIFACT_ROOT = Path(
    os.environ.get("PAPER_SUMMARY_ARTIFACT_ROOT", str(default_artifact_root))
).expanduser().resolve()
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print("Paper-summary source:", SOURCE_DIR)
print("Persistent artifact root:", ARTIFACT_ROOT)


Mounted at /content/drive
Paper-summary source: /content/nsbi-lhc-toolkit/workshops/ml4hep_tifr_colab/paper_summary
Persistent artifact root: /content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP


In [ ]:
from config import (
    DEFAULT_ML_SEEDS,
    PAPER_BUDGETS,
    SMOKE_BUDGETS,
    SMOKE_ML_SEEDS,
    campaign_config,
    campaign_signature,
)

PROFILE = os.environ.get("PAPER_SUMMARY_PROFILE", "PAPER").upper()
CAMPAIGN_BUDGETS = list(PAPER_BUDGETS if PROFILE == "PAPER" else SMOKE_BUDGETS)
CAMPAIGN_ML_SEEDS = list(DEFAULT_ML_SEEDS if PROFILE == "PAPER" else SMOKE_ML_SEEDS)

def execution_subset(environment_name, configured):
    raw = os.environ.get(environment_name, "").strip()
    values = list(configured) if not raw else [int(value) for value in raw.split(",")]
    unknown = set(values) - set(configured)
    if not values or unknown:
        raise ValueError(f"Invalid {environment_name}: {values}; unknown={sorted(unknown)}")
    return values

BUDGETS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_BUDGETS", CAMPAIGN_BUDGETS)
ML_SEEDS_TO_RUN = execution_subset("PAPER_SUMMARY_RUN_SEEDS", CAMPAIGN_ML_SEEDS)
LOAD_IF_AVAILABLE = os.environ.get("PAPER_SUMMARY_LOAD_IF_AVAILABLE", "1") != "0"

CAMPAIGN = campaign_config(profile=PROFILE)
print(json.dumps({
    "profile": PROFILE,
    "campaign_budgets": CAMPAIGN_BUDGETS,
    "campaign_ml_seeds": CAMPAIGN_ML_SEEDS,
    "budgets_to_run": BUDGETS_TO_RUN,
    "ml_seeds_to_run": ML_SEEDS_TO_RUN,
    "load_if_available": LOAD_IF_AVAILABLE,
    "campaign_signature": campaign_signature(CAMPAIGN),
}, indent=2))


{
  "profile": "PAPER",
  "campaign_budgets": [
    10000,
    100000,
    1000000
  ],
  "campaign_ml_seeds": [
    31082026,
    31082027,
    31082028
  ],
  "budgets_to_run": [
    10000,
    100000,
    1000000
  ],
  "ml_seeds_to_run": [
    31082026,
    31082027,
    31082028
  ],
  "load_if_available": true,
  "campaign_signature": "sha256-e455fa167513"
}


In [ ]:
RUN_EXACT_JANA_PAPER = True
RUN_NOMINAL_MATCHED = True
INSTALL_EXACT_JANA_ENV_IF_MISSING = (
    os.environ.get("PAPER_SUMMARY_INSTALL_JANA_ENV", "1") != "0"
)


## GPU execution and interruption recovery

Select a **GPU** in Colab's Runtime settings before running this notebook.
The next cell installs CUDA 11.8/cuDNN 8.6 libraries in the isolated Python
3.11 environment and verifies GPU computation with TensorFlow 2.12. It does
not replace the modern notebook's PyTorch/CUDA installation. CPU fallback is
not allowed for exact-JANA training in this notebook.

The 1,000,000-simulation exact-JANA runs use **batch size 1024**; the 10k and
100k runs retain batch size 32. The 1M runs still use 100 epochs, now with
977 updates per epoch (including the final partial batch), and Adam's cosine
learning-rate decay from $5 \times 10^{-4}$ to zero spans those 97,700 updates.
This is a documented change to the upstream optimization protocol, not a
claim of identical convergence. The saved training contract records the
actual batch size. Old batch-32 checkpoints must not be resumed as batch-1024
runs; after the one-time reset, subsequent interruptions resume normally.

The PAPER grid again requires **three independent ML seeds at every budget**,
including 1,000,000. Leave `LOAD_IF_AVAILABLE=True`: completed smaller-budget
results are reused, missing runs train, and interrupted runs resume from the
last completed epoch. Keep the same Drive artifact root. Each run writes
`resume/` checkpoints and `training_progress.json` inside its training folder.
Model weights, Adam slots/iterations, and the completed epoch are restored;
the cosine schedule remains the full 100-epoch schedule for that batch size. An unfinished
epoch is repeated. Resume does not promise bitwise-identical dropout/shuffling
to an uninterrupted run.

Training prints the GPU, epoch starts/ends, batch progress about once a minute,
losses and checkpoint locations. The most recent two checkpoint generations
are retained, along with all epoch loss histories. Concurrent notebooks keep
using the existing per-run claims; after a killed session, an abandoned claim
is reclaimed once its existing six-hour lease expires.


In [ ]:
if RUN_EXACT_JANA_PAPER:
    # Reload so rerunning this cell after the setup cell pulls a repository
    # update cannot retain an older helper from the current Colab process.
    import importlib
    import utils_jana
    import utils_jana_runtime
    import utils_jana_gpu
    import utils_jana_checkpoint

    utils_jana = importlib.reload(utils_jana)
    utils_jana_runtime = importlib.reload(utils_jana_runtime)
    utils_jana_gpu = importlib.reload(utils_jana_gpu)
    utils_jana_checkpoint = importlib.reload(utils_jana_checkpoint)

    print("Preparing the isolated exact-JANA runtime (first install can take several minutes).")
    JANA_PYTHON = utils_jana_runtime.ensure_jana_environment(
        ARTIFACT_ROOT,
        install_if_missing=INSTALL_EXACT_JANA_ENV_IF_MISSING,
        require_gpu=True,
    )
    print("Exact-JANA Python:", JANA_PYTHON)


Preparing the isolated exact-JANA runtime (first install can take several minutes).
Rebuilding isolated exact-JANA environment: /content/paper_summary_jana_env
Installing pinned exact-JANA packages into: /content/paper_summary_jana_env
Exact-JANA installation probe: {"base_prefix": "/root/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu", "executable": "/content/paper_summary_jana_env/bin/python", "numpy": "1.23.5", "prefix": "/content/paper_summary_jana_env", "purelib": "/content/paper_summary_jana_env/lib/python3.11/site-packages", "python": "3.11.16", "site_packages": ["/content/paper_summary_jana_env/lib/python3.11/site-packages"]}
[exact JANA GPU] Colab hardware: NVIDIA A100-SXM4-40GB, 580.82.07
[exact JANA GPU] Installing/checking CUDA 11.8 and cuDNN 8.6 in the isolated environment.
Exact-JANA Python: /content/paper_summary_jana_env/bin/python


In [ ]:
from IPython.display import display

def display_result(result):
    if hasattr(result, "style"):
        display(result.style.format(precision=4).hide(axis="index"))
    elif isinstance(result, dict):
        for name, value in result.items():
            print(f"\n{name}")
            if hasattr(value, "style"):
                display(value.style.format(precision=4).hide(axis="index"))
            else:
                display(value)
    else:
        display(result)


In [6]:
import importlib
import utils

# Pull repository fixes into an already-open Colab runtime.
utils = importlib.reload(utils)

JANA_RESULT = utils.run_jana_campaign(
    artifact_root=ARTIFACT_ROOT,
    campaign=CAMPAIGN,
    run_exact_paper=RUN_EXACT_JANA_PAPER,
    run_matched=RUN_NOMINAL_MATCHED,
    budgets_to_run=BUDGETS_TO_RUN,
    ml_seeds_to_run=ML_SEEDS_TO_RUN,
    load_if_available=LOAD_IF_AVAILABLE,
)
display_result(JANA_RESULT)


[exact JANA] Preserved stale evaluation cache for budget=10000/seed=31082026 as standardized.recovery-stale-fb111961a60c; the trained checkpoint is unchanged and will be reused.
[exact JANA] Preserved stale evaluation cache for budget=10000/seed=31082027 as standardized.recovery-stale-79c080c56bf8; the trained checkpoint is unchanged and will be reused.
[exact JANA] Preserved stale evaluation cache for budget=10000/seed=31082028 as standardized.recovery-stale-aafa229e1bb0; the trained checkpoint is unchanged and will be reused.
[exact JANA] Running budget=10000/seed=31082026
[exact JANA GPU] Budgets [10000]: batch size 32.
[exact JANA evaluation] Starting/reusing diagnostics for budget_n0010000/seed_31082026.
/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/evaluation_manifest.json
[exact JANA] Running budget=10000/seed=31082027
[exact JANA GPU] Budgets [10000]: batch size 32.
[exact JANA evaluation]

schema,campaign_signature,method,factorization,budget,simulator_calls,training_rows,validation_rows,ml_seed,observation,posterior_C2ST,posterior_MMD,likelihood_posterior_C2ST,likelihood_posterior_MMD,posterior_likelihood_route_C2ST,posterior_likelihood_route_MMD,posterior_ESS_fraction,posterior_max_weight,likelihood_posterior_ESS_fraction,likelihood_posterior_max_weight,bayes_cycle_pearson,bayes_cycle_slope,bayes_cycle_residual_rms,bayes_cycle_rows,bayes_cycle_theta_fingerprint,likelihood_log_Z_rms,likelihood_log_Z_mean,likelihood_log_Z_max_abs,exact_likelihood_log_error,exact_likelihood_centered_log_error,exact_likelihood_rows,likelihood_audit_rows,likelihood_audit_fingerprint,audit_bank_fingerprint,deployed_proposal_exact_likelihood_log_error,deployed_proposal_exact_likelihood_centered_log_error,deployed_proposal_exact_likelihood_rows,deployed_proposal_bayes_cycle_pearson,deployed_proposal_bayes_cycle_slope,deployed_proposal_bayes_cycle_residual_rms,deployed_proposal_bayes_cycle_rows,proposal_base_scale,proposal_prior_fraction,posterior_joint_C2ST,posterior_joint_MMD,predictive_x_C2ST,predictive_x_MMD,predictive_joint_C2ST,predictive_joint_MMD,posterior_joint_ESS_fraction,likelihood_joint_ESS_fraction
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows,none,10000,10000,8999,1001,31082026,1,0.9859,0.6657,0.9816,0.6287,0.6451,0.1170,1.0000,0.0001,0.4911,0.0005,0.4320,1.1517,1.6287,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0000,0.0000,0.0000,585900251.2820,585869257.4664,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,104168486050000448.0000,104166568983122048.0000,150000,0.2963,0.8413,1.4989,150000,1.0000,0.0000,0.8312,0.0198,0.7770,0.0348,0.8702,0.0201,1.0000,1.0000
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows,none,10000,10000,8999,1001,31082026,2,0.9779,0.6592,0.9708,0.5718,0.7297,0.2573,1.0000,0.0001,0.1473,0.0019,0.4480,0.4592,1.4013,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0000,0.0000,0.0000,585900251.2820,585869257.4664,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,68684694453122072576.0000,68684392435120324608.0000,150000,0.2617,0.3165,1.4762,150000,1.0000,0.0000,0.8312,0.0198,0.7770,0.0348,0.8702,0.0201,1.0000,1.0000
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows,none,10000,10000,8999,1001,31082026,3,0.9516,0.4760,0.9707,0.5367,0.6873,0.0614,1.0000,0.0001,0.1491,0.0018,0.7657,1.2360,2.1526,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0000,0.0000,0.0000,585900251.2820,585869257.4664,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,7903076212215861248.0000,7903032746568397824.0000,150000,0.7732,0.8933,1.1984,150000,1.0000,0.0000,0.8312,0.0198,0.7770,0.0348,0.8702,0.0201,1.0000,1.0000
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows,none,10000,10000,8999,1001,31082026,4,0.9874,0.7510,0.9874,0.7397,0.7560,0.1616,1.0000,0.0001,0.0308,0.0094,0.5928,2.2495,2.7469,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0000,0.0000,0.0000,585900251.2820,585869257.4664,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,1760588309725379072.0000,1760575064988937728.0000,150000,0.5406,2.3135,2.4445,150000,1.0000,0.0000,0.8312,0.0198,0.7770,0.0348,0.8702,0.0201,1.0000,1.0000
slcp_paper_summary_v2,sha256-e455fa167513,separate_flows,none,10000,10000,8999,1001,31082026,5,0.9466,0.4616,0.9680,0.4887,0.6193,0.0613,1.0000,0.0001,0.1018,0.0066,0.8157,1.0359,1.4589,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,0.0000,0.0000,0.0000,585900251.2820,585869257.4664,10000,10000,64ee79b8785c518ada400f8454ba5d9fd1ba8


jana_paper


schema,campaign_signature,method,factorization,budget,simulator_calls,training_rows,validation_rows,ml_seed,observation,posterior_C2ST,posterior_MMD,likelihood_posterior_C2ST,likelihood_posterior_MMD,posterior_likelihood_route_C2ST,posterior_likelihood_route_MMD,posterior_joint_C2ST,posterior_joint_MMD,predictive_x_C2ST,predictive_x_MMD,predictive_joint_C2ST,predictive_joint_MMD,posterior_joint_ESS_fraction,likelihood_joint_ESS_fraction,posterior_ESS_fraction,posterior_max_weight,likelihood_posterior_ESS_fraction,likelihood_posterior_max_weight,bayes_cycle_pearson,bayes_cycle_slope,bayes_cycle_residual_rms,bayes_cycle_rows,deployed_proposal_bayes_cycle_pearson,deployed_proposal_bayes_cycle_slope,deployed_proposal_bayes_cycle_residual_rms,deployed_proposal_bayes_cycle_rows,likelihood_log_Z_rms,likelihood_log_Z_mean,likelihood_log_Z_max_abs,exact_likelihood_log_error,exact_likelihood_centered_log_error,exact_likelihood_rows,deployed_proposal_exact_likelihood_log_error,deployed_proposal_exact_likelihood_centered_log_error,deployed_proposal_exact_likelihood_rows,independent_audit_exact_likelihood_log_error,independent_audit_exact_likelihood_centered_log_error,independent_audit_exact_likelihood_rows,paper_simulator_calls,simulation_accounting,proposal,proposal_base_scale,proposal_prior_fraction,route_arrays,likelihood_audit_arrays,likelihood_normalization_arrays,audit_row_fingerprint,audit_bank_fingerprint,likelihood_audit_fingerprint,likelihood_audit_rows,bayes_cycle_theta_fingerprint,bayes_cycle_theta_grid_fingerprint
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper,none,10000,10304,10000,300,31082026,1,0.9210,0.5026,0.7659,0.1775,0.8719,0.2738,0.5086,0.0113,0.5163,0.0153,0.5207,0.0145,1.0000,1.0000,1.0000,0.0001,0.0039,0.0300,0.5133,7.1529,208.1445,10000,0.2802,9.1383,44.7679,137662,0.0000,0.0000,0.0000,585900251.4338,585869257.5317,10000,103418222163239072.0000,103417687210606752.0000,150000,585900251.4338,585869257.5317,10000,10304,10000 + 2 + 2 + 300,method_native_nominal_q_phi,1.0000,0.0000,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/observation_01_routes.npz,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/independent_likelihood_audit.npz,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/likelihood_normalization_inputs.npz,1ed45cb1e3701bcc2581a4f74c13b48174e691b5a333a28976d93f8fb933004d,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,64ee79b8785c518ada400f8454ba5d9fd1ba8c7debeffa163e59f8ae45e3da99,10000,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142,da0c134e2d2277f6992373b2678e6a24289d842abb862e9cc20109e91743b142
slcp_paper_summary_v2,sha256-e455fa167513,jana_paper,none,10000,10304,10000,300,31082026,2,0.8976,0.4438,0.6916,0.0949,0.9111,0.3176,0.5086,0.0113,0.5163,0.0153,0.5207,0.0145,1.0000,1.0000,1.0000,0.0001,0.0045,0.0196,0.6139,8.0373,450.7529,10000,0.1998,4.6855,28.8566,110173,0.0000,0.0000,0.0000,585900251.4338,585869257.5317,10000,47516773954738700288.0000,47516614972969246720.0000,150000,585900251.4338,585869257.5317,10000,10304,10000 + 2 + 2 + 300,method_native_nominal_q_phi,1.0000,0.0000,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/observation_02_routes.npz,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/independent_likelihood_audit.npz,/content/drive/MyDrive/hybrid_nsbi_ml/paper_summary_SLCP/results/jana_paper/sha256-e455fa167513/budget_10000/seed_31082026/standardized/likelihood_normalization_inputs.npz,1ed45cb1e3701bcc2581a4f74c13b48174e691b5a333a28976d93f8fb933004d,77fa56d1974ebac5341b2f710946bca71486e06a026a5bd383ec0055d056452a,64ee79